# Reentrenamiento del modelo con fotos de laboratorio + fotos reales de campo

**Sistema Seguro de Detección de Enfermedades en Plantas mediante IA**

El modelo original se entrenó solo con PlantVillage (fotos de laboratorio, fondo gris). Con fotos reales acierta muy poco.
Este notebook parte de **tu mismo `modelo_plantas.keras`** y lo ajusta con:

| Dataset | Tipo de foto | Uso |
|---|---|---|
| PlantVillage (Kaggle `emmarex/plantdisease`) | Laboratorio | 80 % entrenamiento / 20 % prueba |
| PlantDoc (GitHub `pratikkayal/PlantDoc-Dataset`) | Campo e internet | carpeta `train` para entrenar, carpeta `test` para medir |

Se mantienen las **mismas 15 clases**, la misma arquitectura y los mismos nombres de capas, así el backend sigue funcionando sin cambios.
Tu modelo original **no se sobrescribe**: el nuevo se guarda en `MyDrive/modelo_v2/`.

### Antes de empezar
1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**.
2. Ten en la raíz de tu Google Drive: `modelo_plantas.keras` y `modelo_hash.txt` (los mismos de `backend/modelo/`).
3. Ejecuta las celdas **en orden**, una por una (o *Entorno de ejecución → Ejecutar todas*). Tarda unos 30–45 min con GPU.

In [ ]:
# 1. Comprobar GPU y montar Google Drive
import tensorflow as tf, os, json, hashlib
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "⚠️ NO HAY GPU: activa la GPU T4 en el menú Entorno de ejecución")

from google.colab import drive
drive.mount("/content/drive")

# Rutas (cámbialas si guardaste los archivos en otra carpeta de Drive)
MODELO_ORIGINAL = "/content/drive/MyDrive/modelo_plantas.keras"
HASH_ORIGINAL   = "/content/drive/MyDrive/modelo_hash.txt"
SALIDA          = "/content/drive/MyDrive/modelo_v2"
os.makedirs(SALIDA, exist_ok=True)

IMG_SIZE, BATCH, SEED = (224, 224), 32, 123
CLASES = ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight',
          'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight',
          'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot',
          'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot',
          'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
for p in (MODELO_ORIGINAL, HASH_ORIGINAL):
    print(("✓ " if os.path.exists(p) else "✗ FALTA ") + p)

In [ ]:
# 2. Verificar la integridad del modelo original (SHA-256) antes de usarlo
def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

hash_esperado = open(HASH_ORIGINAL, encoding="utf-8").read().strip().split()[0].lower()
hash_actual = sha256_archivo(MODELO_ORIGINAL)
print("Esperado:", hash_esperado)
print("Actual:  ", hash_actual)
assert hash_actual == hash_esperado, "¡ALERTA! El modelo fue modificado. No se reentrena."
print("✓ Modelo original íntegro")

In [ ]:
# 3. Descargar PlantVillage (el mismo dataset que usaste)
import kagglehub
ruta_kaggle = kagglehub.dataset_download("emmarex/plantdisease")

def find_data_dir(base):
    for root, dirs, files in os.walk(base):
        if "PlantVillage" in dirs:
            pv = os.path.join(root, "PlantVillage")
            anidado = os.path.join(pv, "PlantVillage")
            return anidado if os.path.isdir(anidado) else pv
    return base

PV_DIR = find_data_dir(ruta_kaggle)
print("PlantVillage en:", PV_DIR)
print(sorted(os.listdir(PV_DIR)) == CLASES and "✓ Las 15 clases coinciden" or sorted(os.listdir(PV_DIR)))

In [ ]:
# 4. Descargar PlantDoc (fotos reales de campo e internet)
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/PlantDoc
PD_DIR = "/content/PlantDoc"

# Carpetas de PlantDoc → tus 15 clases (las demás plantas se ignoran)
MAPA_PLANTDOC = {
    "Bell_pepper leaf":                      "Pepper__bell___healthy",
    "Bell_pepper leaf spot":                 "Pepper__bell___Bacterial_spot",
    "Potato leaf early blight":              "Potato___Early_blight",
    "Potato leaf late blight":               "Potato___Late_blight",
    "Tomato leaf":                           "Tomato_healthy",
    "Tomato Early blight leaf":              "Tomato_Early_blight",
    "Tomato leaf late blight":               "Tomato_Late_blight",
    "Tomato mold leaf":                      "Tomato_Leaf_Mold",
    "Tomato Septoria leaf spot":             "Tomato_Septoria_leaf_spot",
    "Tomato leaf bacterial spot":            "Tomato_Bacterial_spot",
    "Tomato leaf mosaic virus":              "Tomato__Tomato_mosaic_virus",
    "Tomato leaf yellow virus":              "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato two spotted spider mites leaf":  "Tomato_Spider_mites_Two_spotted_spider_mite",
}
print(sorted(os.listdir(f"{PD_DIR}/train")))

In [ ]:
# 5. Armar las listas de imágenes (descartando archivos dañados) y los conjuntos de entrenamiento y prueba
import numpy as np, random
from PIL import Image
EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def imagen_valida(ruta):
    try:
        with Image.open(ruta) as im:
            im.convert("RGB")
        return True
    except Exception:
        return False

def listar(carpeta_base, mapa=None):
    rutas, etiquetas = [], []
    for carpeta in sorted(os.listdir(carpeta_base)):
        clase = mapa.get(carpeta) if mapa else carpeta
        if clase not in CLASES:
            continue
        for f in sorted(os.listdir(os.path.join(carpeta_base, carpeta))):
            ruta = os.path.join(carpeta_base, carpeta, f)
            if f.lower().endswith(EXT) and imagen_valida(ruta):
                rutas.append(ruta); etiquetas.append(CLASES.index(clase))
    return np.array(rutas), np.array(etiquetas)

# PlantVillage: 80/20 estratificado por clase
pv_rutas, pv_y = listar(PV_DIR)
rng = np.random.default_rng(SEED)
es_prueba = np.zeros(len(pv_y), bool)
for c in range(len(CLASES)):
    idx = np.where(pv_y == c)[0]; rng.shuffle(idx)
    es_prueba[idx[: int(len(idx) * 0.2)]] = True
pv_tr, pv_te = (pv_rutas[~es_prueba], pv_y[~es_prueba]), (pv_rutas[es_prueba], pv_y[es_prueba])

# PlantDoc: su propia división train / test
pd_tr = listar(f"{PD_DIR}/train", MAPA_PLANTDOC)
pd_te = listar(f"{PD_DIR}/test", MAPA_PLANTDOC)

print(f"{'Clase':45s} {'PV entr':>8s} {'PV prueba':>9s} {'PD entr':>8s} {'PD prueba':>9s}")
for i, c in enumerate(CLASES):
    print(f"{c:45s} {(pv_tr[1]==i).sum():8d} {(pv_te[1]==i).sum():9d} {(pd_tr[1]==i).sum():8d} {(pd_te[1]==i).sum():9d}")
print("Totales:", len(pv_tr[1]), len(pv_te[1]), len(pd_tr[1]), len(pd_te[1]))

In [ ]:
# 6. Pipelines de datos (tf.data). Las fotos de campo se muestrean con más frecuencia para que pesen en el entrenamiento.
AUTOTUNE = tf.data.AUTOTUNE

def cargar(ruta, y):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)          # 0–255; el modelo ya incluye preprocess_input
    return img, y

def aumentar_color(img, y):
    # Variaciones de luz y color típicas de fotos de celular en campo
    img = tf.image.random_brightness(img, 40.0)
    img = tf.image.random_contrast(img, 0.75, 1.25)
    img = tf.image.random_saturation(img, 0.75, 1.25)
    return tf.clip_by_value(img, 0.0, 255.0), y

def ds_entrenamiento(rutas, y):
    return (tf.data.Dataset.from_tensor_slices((rutas, y)).shuffle(len(y), seed=SEED).repeat()
            .map(cargar, num_parallel_calls=AUTOTUNE).map(aumentar_color, num_parallel_calls=AUTOTUNE))

def ds_prueba(rutas, y):
    return tf.data.Dataset.from_tensor_slices((rutas, y)).map(cargar, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

PESO_CAMPO = 0.35   # 35 % de cada lote son fotos de campo (PlantDoc), 65 % de laboratorio
train_ds = (tf.data.Dataset.sample_from_datasets(
                [ds_entrenamiento(*pv_tr), ds_entrenamiento(*pd_tr)],
                weights=[1 - PESO_CAMPO, PESO_CAMPO], seed=SEED)
            .batch(BATCH).prefetch(AUTOTUNE))
pv_test_ds, pd_test_ds = ds_prueba(*pv_te), ds_prueba(*pd_te)
PASOS_POR_EPOCA = (len(pv_tr[1]) + len(pd_tr[1])) // BATCH
print("Pasos por época:", PASOS_POR_EPOCA)

In [ ]:
# 7. Cargar tu modelo original y medir cómo le va ANTES de reentrenar
model = tf.keras.models.load_model(MODELO_ORIGINAL)
nombres_capas = [l.name for l in model.layers]
print(nombres_capas)
assert "global_average_pooling2d" in nombres_capas, "El backend necesita la capa global_average_pooling2d"

def evaluar(m, ds):
    y_true, y_prob = [], []
    for x, y in ds:
        y_true.append(y.numpy()); y_prob.append(m.predict(x, verbose=0))
    return np.concatenate(y_true), np.concatenate(y_prob)

antes = {nombre: evaluar(model, ds) for nombre, ds in [("laboratorio", pv_test_ds), ("campo", pd_test_ds)]}
for nombre, (yt, yp) in antes.items():
    print(f"ANTES · {nombre}: accuracy {(yp.argmax(1) == yt).mean()*100:.1f} %")

In [ ]:
# 8. Etapa 1: entrenar solo la capa final (la base MobileNetV2 congelada)
base = model.get_layer("mobilenetv2_1.00_224")
base.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist1 = model.fit(train_ds, steps_per_epoch=PASOS_POR_EPOCA, epochs=4,
                  validation_data=pd_test_ds)

In [ ]:
# 9. Etapa 2: ajuste fino de las últimas 40 capas de MobileNetV2
#    BatchNormalization se deja congelada para no romper lo aprendido.
base.trainable = True
for capa in base.layers[:-40]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, tf.keras.layers.BatchNormalization):
        capa.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(5e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
parar = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
hist2 = model.fit(train_ds, steps_per_epoch=PASOS_POR_EPOCA, epochs=12,
                  validation_data=pd_test_ds, callbacks=[parar])

In [ ]:
# 10. Medir DESPUÉS y comparar (laboratorio y campo)
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

despues = {nombre: evaluar(model, ds) for nombre, ds in [("laboratorio", pv_test_ds), ("campo", pd_test_ds)]}
metricas = {}
for nombre in ("laboratorio", "campo"):
    yt, yp_a = antes[nombre]; _, yp_d = despues[nombre]
    pred = yp_d.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(yt, pred, average="weighted", zero_division=0)
    metricas[nombre] = {"n": int(len(yt)),
                        "accuracy_antes": float((yp_a.argmax(1) == yt).mean()),
                        "accuracy": float((pred == yt).mean()),
                        "precision": float(p), "recall": float(r), "f1": float(f1),
                        "matriz_confusion": confusion_matrix(yt, pred, labels=range(len(CLASES))).tolist()}
    print(f"\n===== {nombre.upper()} =====  antes {metricas[nombre]['accuracy_antes']*100:.1f} %  →  después {metricas[nombre]['accuracy']*100:.1f} %")
    print(classification_report(yt, pred, labels=range(len(CLASES)), target_names=CLASES, zero_division=0))

fig, ejes = plt.subplots(1, 2, figsize=(22, 9))
for eje, nombre in zip(ejes, ("laboratorio", "campo")):
    sns.heatmap(np.array(metricas[nombre]["matriz_confusion"]), annot=True, fmt="d", cmap="Blues", ax=eje,
                xticklabels=[c[:16] for c in CLASES], yticklabels=[c[:16] for c in CLASES], cbar=False)
    eje.set_title(f"Matriz de confusión · {nombre} · acc {metricas[nombre]['accuracy']*100:.1f} %")
    eje.set_xlabel("Predicho"); eje.set_ylabel("Real")
plt.tight_layout(); plt.savefig(f"{SALIDA}/matrices_confusion.png", dpi=150); plt.show()

h = {k: hist1.history[k] + hist2.history[k] for k in ("accuracy", "val_accuracy")}
plt.figure(figsize=(8, 4)); plt.plot(h["accuracy"], marker="o", label="Entrenamiento (mixto)")
plt.plot(h["val_accuracy"], marker="o", label="Prueba de campo (PlantDoc)")
plt.axvline(len(hist1.history["accuracy"]) - 0.5, ls="--", c="gray"); plt.legend(); plt.grid(True)
plt.title("Accuracy por época (línea gris: inicio del ajuste fino)"); plt.savefig(f"{SALIDA}/curvas.png", dpi=150); plt.show()

In [ ]:
# 11. Guardar el modelo nuevo, su hash SHA-256, las clases y las métricas en MyDrive/modelo_v2
ruta_modelo = f"{SALIDA}/modelo_plantas.keras"
model.save(ruta_modelo)

nuevo_hash = sha256_archivo(ruta_modelo)
with open(f"{SALIDA}/modelo_hash.txt", "w", encoding="utf-8") as f:
    f.write(nuevo_hash + "\n")
with open(f"{SALIDA}/clases.json", "w", encoding="utf-8") as f:
    json.dump(CLASES, f, indent=4, ensure_ascii=False)
with open(f"{SALIDA}/metricas_v2.json", "w", encoding="utf-8") as f:
    json.dump({"clases": CLASES, "hash_modelo": nuevo_hash, "hash_modelo_original": hash_actual, **metricas}, f, indent=2)

# Comprobación final: el archivo guardado carga bien, mantiene la capa que usa el backend y su hash coincide
m2 = tf.keras.models.load_model(ruta_modelo)
assert "global_average_pooling2d" in [l.name for l in m2.layers]
assert sha256_archivo(ruta_modelo) == nuevo_hash
print("✓ Modelo nuevo guardado:", ruta_modelo)
print("✓ SHA-256:", nuevo_hash)
print("Archivos en", SALIDA, "→", os.listdir(SALIDA))

## Después de entrenar

1. Descarga de `MyDrive/modelo_v2/` estos 3 archivos: `modelo_plantas.keras`, `modelo_hash.txt` y `clases.json`.
2. **Guarda una copia** de tu `backend\modelo\` actual (por ejemplo, renómbrala a `modelo_original`).
3. Copia los 3 archivos nuevos a `C:\laragon\www\Sistema_Detec_Enferm_IA\backend\modelo\`, reemplazando los anteriores.
4. Comprueba en PowerShell:
   ```
   cd C:\laragon\www\Sistema_Detec_Enferm_IA
   python entrenamiento\generar_hash.py --verificar
   cd backend
   python -m pytest -q
   ```
5. Envíame **`metricas_v2.json`** (y si quieres `matrices_confusion.png`) y actualizo la página *Modelo y métricas* del sistema y `entrenamiento/metricas.md` con los números nuevos.